# Detección de fraude en transacciones con tarjeta

Modelo de *machine learning* para detectar transacciones fraudulentas a partir del dataset
[IEEE-CIS Fraud Detection](https://www.kaggle.com/c/ieee-fraud-detection) (~590.000 transacciones reales).

**Enfoque del proyecto:**

1. Unión de las tablas de transacción e identidad de dispositivo.
2. Análisis exploratorio de los patrones asociados al fraude (dispositivo, email, horario, distancia, importe).
3. Entrenamiento de un clasificador **XGBoost** con validación temporal: entrenar con el pasado, validar con el futuro.
4. Interpretación del modelo con **SHAP** a nivel global y local.

In [ ]:
# Descarga del dataset IEEE-CIS Fraud Detection desde Kaggle.
# En Colab: cuando aparezca el botón, sube tu kaggle.json
# (Kaggle -> Settings -> API -> Create New Token).
!pip install opendatasets shap xgboost --quiet

# (Solo en Colab) subir kaggle.json para autenticarse sin teclear credenciales.
try:
    from google.colab import files
    files.upload()
except Exception:
    pass

import opendatasets as od

DATASET_URL = "https://www.kaggle.com/c/ieee-fraud-detection"
od.download(DATASET_URL)

In [ ]:
import os
import gc
import pandas as pd

DATA_FOLDER = "ieee-fraud-detection"

# Tabla de transacciones: importe, tarjeta, email, marca de tiempo.
df_trans = pd.read_csv(os.path.join(DATA_FOLDER, "train_transaction.csv"))

# Tabla de identidad: dispositivo, sistema operativo, navegador.
df_id = pd.read_csv(os.path.join(DATA_FOLDER, "train_identity.csv"))

# Left join sobre TransactionID: conservamos todas las transacciones y
# añadimos la información de dispositivo cuando está disponible.
df = pd.merge(df_trans, df_id, on="TransactionID", how="left")
print(f"Transacciones: {df_trans.shape}")
print(f"Identidad:     {df_id.shape}")
print(f"Dataset final: {df.shape}")

# Optimización de memoria: float64 -> float32 (reduce ~50% de RAM).
float_cols = df.select_dtypes(include=["float64"]).columns
df[float_cols] = df[float_cols].astype("float32")
del df_trans, df_id
gc.collect()

df.head()

## Análisis exploratorio

Buscamos señales asociadas al fraude: tipo de dispositivo, dominio de email, tipo de tarjeta,
hora del día, distancia entre IP y dirección de facturación, y desviación del importe respecto
al gasto habitual de cada tarjeta.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tasa de fraude por modelo de dispositivo.
# Filtramos dispositivos con menos de 100 transacciones para evitar ruido estadístico.
device_stats = df.groupby("DeviceInfo")["isFraud"].agg(fraud_rate="mean", n="count")
device_stats = device_stats[device_stats["n"] > 100].sort_values("fraud_rate", ascending=False)

print("Dispositivos con mayor tasa de fraude:")
display(device_stats.head(10))

plt.figure(figsize=(12, 6))
sns.barplot(x=device_stats.head(10)["fraud_rate"], y=device_stats.head(10).index, palette="Reds_r")
plt.title("Tasa de fraude por modelo de dispositivo")
plt.xlabel("Tasa de fraude")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de fraude por dominio de email (mínimo 500 transacciones).
email_stats = df.groupby("P_emaildomain")["isFraud"].agg(fraud_rate="mean", n="count")
email_stats = email_stats[email_stats["n"] > 500].sort_values("fraud_rate", ascending=False)

print("Dominios de email con mayor tasa de fraude:")
display(email_stats.head(10))

# Tasa de fraude por tipo de tarjeta (crédito vs débito).
card_stats = (
    df.groupby("card6")["isFraud"].agg(fraud_rate="mean", n="count")
    .sort_values("fraud_rate", ascending=False)
)
print("\nTasa de fraude por tipo de tarjeta:")
display(card_stats)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=email_stats.head(5)["fraud_rate"], y=email_stats.head(5).index, ax=ax[0], palette="viridis")
ax[0].set_title("Dominios de email con mayor fraude")
ax[0].set_xlabel("Tasa de fraude")
ax[0].set_ylabel("")

sns.barplot(x=card_stats.index, y=card_stats["fraud_rate"], ax=ax[1], palette="magma")
ax[1].set_title("Tasa de fraude por tipo de tarjeta")
ax[1].set_ylabel("Tasa de fraude")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- Importe de la transacción ---
print("Estadísticos del importe por clase:")
print(df.groupby("isFraud")["TransactionAmt"].describe())

plt.figure(figsize=(10, 4))
sns.histplot(data=df, x="TransactionAmt", hue="isFraud", element="step",
             stat="density", common_norm=False, log_scale=True)
plt.title("Distribución del importe por clase (escala logarítmica)")
plt.tight_layout()
plt.show()

# --- Hora del día ---
# TransactionDT está en segundos desde un origen desconocido.
# Derivamos la hora del día con aritmética modular.
df["Hour"] = (df["TransactionDT"] // 3600) % 24
fraud_by_hour = df.groupby("Hour")["isFraud"].mean()

plt.figure(figsize=(10, 4))
sns.lineplot(x=fraud_by_hour.index, y=fraud_by_hour.values, color="red", marker="o")
plt.title("Tasa de fraude por hora del día")
plt.xlabel("Hora")
plt.ylabel("Tasa de fraude")
plt.grid(True, linestyle="--")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

# --- Sistema operativo ---
# Agrupamos versiones concretas en familias (p. ej. "iOS 11.1" -> "iOS").
def simplify_os(value):
    if pd.isna(value):
        return "Unknown"
    for name in ("Windows", "iOS", "Mac", "Android", "Linux"):
        if name in value:
            return "Mac OS" if name == "Mac" else name
    return "Other"

df["OS_Simple"] = df["id_30"].apply(simplify_os)
os_stats = (
    df.groupby("OS_Simple")["isFraud"].agg(fraud_rate="mean", n="count")
    .sort_values("fraud_rate", ascending=False)
)
print("Tasa de fraude por sistema operativo:")
display(os_stats)

plt.figure(figsize=(10, 4))
sns.barplot(x=os_stats.index, y=os_stats["fraud_rate"], palette="magma")
plt.title("Tasa de fraude por sistema operativo")
plt.ylabel("Tasa de fraude")
plt.tight_layout()
plt.show()

In [ ]:
# Distancia entre IP y dirección de facturación (dist1).
def categorize_distance(d):
    if pd.isna(d):
        return "Desconocida"
    if d < 20:
        return "< 20 mi"
    if d < 100:
        return "20-100 mi"
    if d < 500:
        return "100-500 mi"
    return "> 500 mi"

df["Distancia_Cat"] = df["dist1"].apply(categorize_distance)
dist_stats = (
    df.groupby("Distancia_Cat")["isFraud"].agg(fraud_rate="mean", n="count")
    .sort_values("fraud_rate", ascending=False)
)
print("Tasa de fraude por distancia:")
display(dist_stats)

plt.figure(figsize=(10, 5))
sns.barplot(x=dist_stats["fraud_rate"], y=dist_stats.index, palette="coolwarm")
plt.title("Tasa de fraude por distancia física")
plt.xlabel("Tasa de fraude")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Desviación del importe respecto al gasto histórico de cada tarjeta.
# Análisis exploratorio: aquí usamos el dataset completo solo para visualizar el patrón.
# (En el modelo, esta variable se recalcula sin fuga de datos: ver sección "Modelo".)
card_mean = df.groupby("card1")["TransactionAmt"].transform("mean")
df["Factor_Susto"] = df["TransactionAmt"] / card_mean

print("Desviación del importe respecto a la media de la tarjeta, por clase:")
print(df.groupby("isFraud")["Factor_Susto"].describe())

plt.figure(figsize=(10, 5))
sns.boxplot(x="isFraud", y="Factor_Susto", data=df[df["Factor_Susto"] < 10], palette="coolwarm")
plt.title("Desviación del importe frente a la media histórica de la tarjeta")
plt.xlabel("isFraud (0 = legítima, 1 = fraude)")
plt.ylabel("Importe / media histórica de la tarjeta")
plt.tight_layout()
plt.show()

## Modelo

Entrenamos un **XGBoost** respetando el orden temporal: el 80 % más antiguo de las transacciones
se usa para entrenar y el 20 % más reciente para validar. Esto evita filtrar información del futuro
y reproduce el escenario real de producción, donde el modelo solo conoce el pasado.

In [ ]:
import gc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

# --- Variables temporales ---
df["Hour"] = (df["TransactionDT"] // 3600) % 24
df["Day"] = (df["TransactionDT"] // (3600 * 24)) % 7

# --- Split temporal: pasado para entrenar, futuro para validar ---
# En detección de fraude no se debe barajar: hay que respetar el orden temporal.
df = df.sort_values("TransactionDT").reset_index(drop=True)
split_idx = int(len(df) * 0.8)

# --- Desviación de gasto SIN fuga de datos ---
# La media por tarjeta se calcula SOLO con el tramo de entrenamiento y se aplica
# a ambos tramos, para no filtrar información del futuro en el conjunto de validación.
card_mean_train = df.iloc[:split_idx].groupby("card1")["TransactionAmt"].mean()
global_mean_train = df.iloc[:split_idx]["TransactionAmt"].mean()
df["Media_Tarjeta"] = df["card1"].map(card_mean_train).fillna(global_mean_train)
df["Factor_Susto"] = df["TransactionAmt"] / df["Media_Tarjeta"]

# --- Codificación de variables categóricas ---
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str).fillna("Unknown"))

# --- Matriz de features en float32 (mitad de memoria) ---
# No rellenamos los nulos numéricos: XGBoost los gestiona de forma nativa como "missing".
X = df.drop(["isFraud", "TransactionID", "TransactionDT"], axis=1).astype(np.float32)
y = df["isFraud"]
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

# Liberamos la memoria de los dataframes grandes antes de entrenar.
del df, X
gc.collect()

print(f"Entrenamiento: {len(X_train):>7} transacciones | tasa de fraude {y_train.mean():.3%}")
print(f"Validación:    {len(X_test):>7} transacciones | tasa de fraude {y_test.mean():.3%}")

# --- Entrenamiento ---
clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    eval_metric="auc",
    random_state=42,
)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

# --- Evaluación ---
preds = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, preds)
ap = average_precision_score(y_test, preds)
print(f"\nROC-AUC: {auc:.4f}")
print(f"PR-AUC:  {ap:.4f}  (más informativa que ROC-AUC con clases desbalanceadas)")

# --- Importancia de variables ---
plt.figure(figsize=(10, 8))
xgb.plot_importance(clf, max_num_features=15, height=0.5, importance_type="gain", color="teal")
plt.title("Top 15 variables por importancia (gain)")
plt.tight_layout()
os.makedirs("assets", exist_ok=True)
plt.savefig("assets/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Evaluación

Además del ROC-AUC, analizamos la matriz de confusión y la separación de las distribuciones de
*score*. Después estudiamos cómo el **umbral de decisión** desplaza el equilibrio entre fraudes
detectados y falsos positivos: un problema de negocio, no solo estadístico.

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix

plt.figure(figsize=(18, 5))

# Curva ROC.
plt.subplot(1, 3, 1)
fpr, tpr, _ = roc_curve(y_test, preds)
plt.plot(fpr, tpr, color="#FF4500", lw=3, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.title("Curva ROC")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

# Matriz de confusión (umbral 0.5).
plt.subplot(1, 3, 2)
y_pred_decision = (preds > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_decision)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, annot_kws={"size": 14})
plt.title("Matriz de confusión (umbral 0.5)")
plt.xlabel("Predicción")
plt.ylabel("Realidad")
plt.xticks([0.5, 1.5], ["Limpio", "Fraude"])
plt.yticks([0.5, 1.5], ["Limpio", "Fraude"], rotation=0)

# Separación de las distribuciones de score.
plt.subplot(1, 3, 3)
sns.histplot(x=preds, hue=y_test, element="step", stat="density",
             common_norm=False, palette=["#1f77b4", "#d62728"], bins=30)
plt.title("Distribución del score por clase")
plt.xlabel("Probabilidad de fraude predicha")
plt.xlim(0, 1)

plt.tight_layout()
import os; os.makedirs("assets", exist_ok=True)
plt.savefig("assets/dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# Transacciones con mayor probabilidad de fraude estimada.
top_suspicious = X_test.copy()
top_suspicious["prob_fraude_%"] = preds * 100
top_suspicious["fraude_real"] = y_test.values
print("Transacciones más sospechosas del conjunto de validación:")
display(
    top_suspicious[["TransactionAmt", "Hour", "dist1", "Factor_Susto", "prob_fraude_%", "fraude_real"]]
    .sort_values("prob_fraude_%", ascending=False)
    .head(10)
)

In [ ]:
from sklearn.metrics import confusion_matrix

# Comparación de dos umbrales de decisión:
#   0.5 -> neutral.
#   0.1 -> conservador: prioriza no dejar escapar fraude, a costa de más falsos positivos.
def plot_threshold(threshold, ax):
    y_pred = (preds > threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", cbar=False, annot_kws={"size": 14}, ax=ax)
    ax.set_title(f"Umbral {threshold}")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Realidad")
    ax.set_xticklabels(["Limpio", "Fraude"])
    ax.set_yticklabels(["Limpio", "Fraude"])
    detected, missed, false_alarms = cm[1, 1], cm[1, 0], cm[0, 1]
    return detected, missed, false_alarms

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
d1, m1, f1 = plot_threshold(0.5, ax[0])
d2, m2, f2 = plot_threshold(0.1, ax[1])
plt.tight_layout()
import os; os.makedirs("assets", exist_ok=True)
plt.savefig("assets/umbral.png", dpi=150, bbox_inches="tight")
plt.show()

print("Impacto de bajar el umbral de 0.5 a 0.1:")
print(f"  Fraudes detectados:       {d1} -> {d2}  (+{d2 - d1})")
print(f"  Fraudes no detectados:    {m1} -> {m2}")
print(f"  Falsos positivos (coste): {f1} -> {f2}")

## Explicabilidad

**SHAP** permite ver qué variables impulsan cada predicción, tanto a nivel global (qué pesa en
todo el conjunto) como local (por qué una transacción concreta se marca como fraude). En un
contexto bancario, esta trazabilidad es clave para auditar y justificar cada decisión del modelo.

In [ ]:
import shap
import matplotlib.pyplot as plt

# Explicabilidad global: contribución de cada variable al score en una muestra del conjunto.
explainer = shap.TreeExplainer(clf)
sample = X_test.sample(min(2000, len(X_test)), random_state=42)
shap_values = explainer(sample)
shap_values.feature_names = list(X_test.columns)

shap.plots.beeswarm(shap_values, max_display=15, show=False)
import os; os.makedirs("assets", exist_ok=True)
plt.savefig("assets/shap_global.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# Explicabilidad local: descomposición del score de una transacción de alto riesgo (prob > 0.90).
high_risk = X_test[preds > 0.90].iloc[[0]]
explainer = shap.TreeExplainer(clf)
shap_values = explainer(high_risk)

# Traducción de nombres técnicos a etiquetas de negocio.
business_names = {
    "C1": "Frecuencia de cuentas",
    "C2": "Frecuencia de tarjetas",
    "C6": "Frecuencia de clics",
    "C11": "Repetición de identidad",
    "C13": "Comportamiento recurrente",
    "C14": "Velocidad de conexión",
    "card1": "Banco emisor (ID)",
    "card2": "Tipo de tarjeta",
    "card3": "País de la tarjeta",
    "card6": "Crédito / débito",
    "TransactionAmt": "Importe",
    "Factor_Susto": "Desviación de gasto",
    "dist1": "Distancia física (mi)",
    "P_emaildomain": "Dominio de email",
    "Hour": "Hora del día",
    "addr1": "Región (código postal)",
}
shap_values.feature_names = [business_names.get(c, c) for c in X_test.columns]

shap.plots.waterfall(shap_values[0], max_display=12, show=False)
import os; os.makedirs("assets", exist_ok=True)
plt.savefig("assets/shap_local.png", dpi=150, bbox_inches="tight")
plt.show()